# Predicción de intención de compra en sesiones de comercio electrónico

## Descripción del problema

En este trabajo voy a usar un dataset con información sobre sesiones de usuarios en un sitio de comercio electrónico. A partir de distintas características de navegación, voy a intentar predecir si una sesión termina o no en una compra.

## Objetivo

Mi objetivo es comparar dos modelos de clasificación para ver cuál funciona mejor en este problema. La variable que quiero predecir es Revenue, que indica si la sesión terminó en una compra (True) o no (False).
Para hacer la comparación voy a usar dos modelos vistos durante la cursada: Regresión Logística y Árbol de Decisión.

### Fuente del dataset

UCI Machine Learning Repository — *Online Shoppers Purchasing Intention Dataset*.

## Librerías

In [26]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

## Carga y exploración inicial de los datos

In [5]:
df = pd.read_csv("online_shoppers_intention.csv")

df.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [6]:
df.shape

(12330, 18)

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  str    
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType              12330 no

In [8]:
df["Revenue"].value_counts()

Revenue
False    10422
True      1908
Name: count, dtype: int64

In [9]:
df["Revenue"].value_counts(normalize=True) * 100

Revenue
False    84.525547
True     15.474453
Name: proportion, dtype: float64

### Distribución de la variable objetivo

Al revisar `Revenue`, veo que hay un desbalance entre las clases. Aproximadamente el 84,5 % de las sesiones no terminan en una compra (`False`) y el 15,5 % sí terminan en una compra (`True`).

Por eso no voy a mirar solamente la Accuracy, ya que podría dar una impresión engañosa del rendimiento. También voy a tener en cuenta Precision, Recall y F1-score.

In [10]:
df.duplicated().sum()

np.int64(125)

In [11]:
duplicados = df[df.duplicated(keep=False)]

duplicados.head(10)

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
85,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Feb,1,1,1,3,Returning_Visitor,False,False
132,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Feb,3,2,3,3,Returning_Visitor,False,False
158,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Feb,1,1,1,3,Returning_Visitor,False,False
159,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Feb,3,2,3,3,Returning_Visitor,False,False
178,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Feb,3,2,3,3,Returning_Visitor,False,False
252,0,0.0,0,0.0,2,0.0,0.2,0.2,0.0,0.0,Mar,1,1,1,1,Returning_Visitor,False,False
286,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Mar,2,2,1,1,Returning_Visitor,False,False
293,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Mar,1,1,1,1,Returning_Visitor,True,False
298,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Mar,1,1,8,1,Returning_Visitor,False,False
330,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Mar,3,2,3,1,Returning_Visitor,False,False


### Registros duplicados

Encontré 125 filas duplicadas. Como el dataset no tiene un identificador único para cada sesión, no puedo asegurar que sean errores y no sesiones distintas con las mismas características.

Por eso decidí conservarlas.

### Identificación de variables categóricas

In [41]:
columnas_categoricas = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType",
    "Weekend"
]

for columna in columnas_categoricas:
    print(f"{columna}: {df[columna].unique()}")

Month: <ArrowStringArray>
['Feb', 'Mar', 'May', 'Oct', 'June', 'Jul', 'Aug', 'Nov', 'Sep', 'Dec']
Length: 10, dtype: str
OperatingSystems: [1 2 4 3 7 6 8 5]
Browser: [ 1  2  3  4  5  6  7 10  8  9 12 13 11]
Region: [1 9 2 3 4 5 6 7 8]
TrafficType: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 18 19 16 17 20]
VisitorType: <ArrowStringArray>
['Returning_Visitor', 'New_Visitor', 'Other']
Length: 3, dtype: str
Weekend: [False  True]


### Identificación de variables numéricas

### Selección de variables

Decidí no usar `PageValues` porque está muy relacionada con las transacciones del sitio. Para evitar que el modelo use información demasiado cercana a la variable objetivo `Revenue`, prefiero dejarla fuera.

In [40]:
columnas_numericas = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "SpecialDay"
]

print("Variables numéricas:", columnas_numericas)
print("Variables categóricas:", columnas_categoricas)

Variables numéricas: ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'SpecialDay']
Variables categóricas: ['Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend']


## Preparación de los datos

In [55]:
X = df.drop(["Revenue", "PageValues"], axis=1)
y = df["Revenue"]

print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)

Dimensiones de X: (12330, 16)
Dimensiones de y: (12330,)


### División en entrenamiento y prueba

In [56]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

Entrenamiento: (9864, 16)
Prueba: (2466, 16)


In [57]:
print("Distribución en entrenamiento:")
print(y_train.value_counts(normalize=True) * 100)

print("\nDistribución en prueba:")
print(y_test.value_counts(normalize=True) * 100)

Distribución en entrenamiento:
Revenue
False    84.529603
True     15.470397
Name: proportion, dtype: float64

Distribución en prueba:
Revenue
False    84.509327
True     15.490673
Name: proportion, dtype: float64


### Codificación de variables categóricas

In [58]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_train_cat = encoder.fit_transform(X_train[columnas_categoricas])
X_test_cat = encoder.transform(X_test[columnas_categoricas])

X_train_cat = pd.DataFrame(
    X_train_cat,
    columns=encoder.get_feature_names_out(columnas_categoricas),
    index=X_train.index
)

X_test_cat = pd.DataFrame(
    X_test_cat,
    columns=encoder.get_feature_names_out(columnas_categoricas),
    index=X_test.index
)

print("Variables categóricas antes:", len(columnas_categoricas))
print("Columnas después del One-Hot Encoding:", X_train_cat.shape[1])

Variables categóricas antes: 7
Columnas después del One-Hot Encoding: 65


### Escalado de variables numéricas

In [59]:
scaler = StandardScaler()

X_train_num_scaled = scaler.fit_transform(X_train[columnas_numericas])
X_test_num_scaled = scaler.transform(X_test[columnas_numericas])

X_train_num_scaled = pd.DataFrame(
    X_train_num_scaled,
    columns=columnas_numericas,
    index=X_train.index
)

X_test_num_scaled = pd.DataFrame(
    X_test_num_scaled,
    columns=columnas_numericas,
    index=X_test.index
)

print("Entrenamiento numérico escalado:", X_train_num_scaled.shape)
print("Prueba numérica escalada:", X_test_num_scaled.shape)

Entrenamiento numérico escalado: (9864, 9)
Prueba numérica escalada: (2466, 9)


### Datos preparados para Regresión Logística

In [60]:
X_train_log = pd.concat(
    [X_train_num_scaled, X_train_cat],
    axis=1
)

X_test_log = pd.concat(
    [X_test_num_scaled, X_test_cat],
    axis=1
)

print("Entrenamiento:", X_train_log.shape)
print("Prueba:", X_test_log.shape)

Entrenamiento: (9864, 74)
Prueba: (2466, 74)


## Regresión Logística

In [61]:
modelo_logistico = LogisticRegression(max_iter=1000)

modelo_logistico.fit(X_train_log, y_train)

y_pred_log = modelo_logistico.predict(X_test_log)

print("Modelo de Regresión Logística entrenado correctamente.")

Modelo de Regresión Logística entrenado correctamente.


### Evaluación de la Regresión Logística

In [62]:
accuracy_log = accuracy_score(y_test, y_pred_log)
precision_log = precision_score(y_test, y_pred_log)
recall_log = recall_score(y_test, y_pred_log)
f1_log = f1_score(y_test, y_pred_log)

print("Accuracy:", accuracy_log)
print("Precision:", precision_log)
print("Recall:", recall_log)
print("F1-score:", f1_log)

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred_log))

Accuracy: 0.8446877534468775
Precision: 0.48148148148148145
Recall: 0.034031413612565446
F1-score: 0.06356968215158924

Matriz de confusión:
[[2070   14]
 [ 369   13]]


### Interpretación  de la Regresión Logística

La Regresión Logística obtuvo una Accuracy del 84,5 %, pero un Recall de solo 3,4 %. Esto significa que, aunque acierta en gran parte de los casos, logra detectar muy pocas de las sesiones que realmente terminan en una compra.

## Árbol de Decisión

In [63]:
X_train_tree = pd.concat(
    [X_train[columnas_numericas], X_train_cat],
    axis=1
)

X_test_tree = pd.concat(
    [X_test[columnas_numericas], X_test_cat],
    axis=1
)

print("Entrenamiento:", X_train_tree.shape)
print("Prueba:", X_test_tree.shape)

Entrenamiento: (9864, 74)
Prueba: (2466, 74)


In [64]:
modelo_arbol = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

modelo_arbol.fit(X_train_tree, y_train)

y_pred_tree = modelo_arbol.predict(X_test_tree)

print("Modelo de Árbol de Decisión entrenado correctamente.")

Modelo de Árbol de Decisión entrenado correctamente.


### Evaluación del Árbol de Decisión

In [65]:
accuracy_tree = accuracy_score(y_test, y_pred_tree)
precision_tree = precision_score(y_test, y_pred_tree)
recall_tree = recall_score(y_test, y_pred_tree)
f1_tree = f1_score(y_test, y_pred_tree)

print("Accuracy:", accuracy_tree)
print("Precision:", precision_tree)
print("Recall:", recall_tree)
print("F1-score:", f1_tree)

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred_tree))

Accuracy: 0.8438767234387672
Precision: 0.4794520547945205
Recall: 0.09162303664921466
F1-score: 0.15384615384615385

Matriz de confusión:
[[2046   38]
 [ 347   35]]


### Interpretación del Árbol de Decisión

El Árbol de Decisión obtuvo una Accuracy del 84,4 % y un Recall del 9,2 %. Aunque sigue teniendo dificultades para detectar las compras, logró identificar más casos positivos que la Regresión Logística.

## Comparación de modelos

In [66]:
comparacion = pd.DataFrame({
    "Modelo": ["Regresión Logística", "Árbol de Decisión"],
    "Accuracy": [accuracy_log, accuracy_tree],
    "Precision": [precision_log, precision_tree],
    "Recall": [recall_log, recall_tree],
    "F1-score": [f1_log, f1_tree]
})

comparacion

,Modelo,Accuracy,Precision,Recall,F1-score
0,Regresión Logística,0.844688,0.481481,0.034031,0.063570
1,Árbol de Decisión,0.843877,0.479452,0.091623,0.153846


### Comparación y conclusión

Los dos modelos tuvieron resultados parecidos en Accuracy y Precision, pero ambos tuvieron un Recall bajo.
Entre los dos, yo elegiría el Árbol de Decisión porque logró detectar más compras reales y obtuvo un F1-score mayor.
De todas formas, considero que con esta configuración ninguno de los dos logró buenos resultados para detectar las sesiones que realmente terminan en una compra.